In [ ]:
!pip install quandl
!pip install PyPortfolioOpt==1.5.5
!pip install arch
!pip install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 985.1/985.1 kB 18.5 MB/s eta 0:00:00


In [ ]:
!pip install scikit-learn==1.2.2
import sklearn
import pypfopt
print(sklearn.__version__)
print(pypfopt.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 41.8 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.5.2
    Uninstalling scikit-learn-1.5.2:
      Successfully uninstalled scikit-learn-1.5.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlxtend 0.23.3 requires scikit-learn>=1.3.1, but you have scikit-learn 1.2.2 which is incompatible.


'1.2.2'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime as dt
import datetime
import quandl
import math
import pypfopt
import random
from statsmodels.stats.diagnostic import het_arch
from arch import arch_model
from pypfopt import EfficientFrontier,objective_functions
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn import metrics
from copy import deepcopy

In [ ]:
pypfopt.__version__

'1.5.5'

In [ ]:
!pip install yfinance==0.2.40
from pandas_datareader import data as pdr
import yfinance as yf
yf.pdr_override()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 2.8 MB/s eta 0:00:00
  Attempting uninstall: yfinance
    Found existing installation: yfinance 0.2.49
    Uninstalling yfinance-0.2.49:
      Successfully uninstalled yfinance-0.2.49
yfinance: pandas_datareader support is deprecated & semi-broken so will be removed in a future verison. Just use yfinance.


In [ ]:
def modify_date(date):
    return date[:10]

In [ ]:
def data_cleaning(data):
    if (data.isna()).any().any():
        missing = np.where(data.isna())
        rows=missing[0]
        cols=missing[1]
        print(rows)
        for row in range(len(rows)):
            col=cols[row]
            if row == len(data)-1:
                data.iloc[row,col]=data.iloc[row-1,col]**2/data.iloc[row-2,col]
            elif row ==0:
                data.iloc[row,col]=data.iloc[row+1,col]**2/data.iloc[row+2,col]
            else:
                data.iloc[row,col]=math.sqrt(data.iloc[row-1,col]*data.iloc[row+1,col])
        print(data)

In [ ]:
def get_price_years_ago(data, date,time):
    year_ago=date-datetime.timedelta(days=int(time*365))
    price_year_ago=data[data.index<=year_ago]
    return price_year_ago

In [ ]:
def max_drawdown_absolute(returns):
    r = returns.add(1).cumprod()
    dd = r.div(r.cummax()).sub(1)
    mdd = dd.min()
    end = dd.argmin()
    start = r.iloc[:end].argmax()
    return abs(mdd), start, end

In [ ]:
def fit_garch(data,p,q):
    lag = max(p,q)
    length =res.shape[0]
    am = arch_model(res, vol='Garch', p=1, o=0, q=1, dist='Normal')
    split_date = one_year_monthly_return.index.values[lag-1]
    results = am.fit(last_obs=split_date)
    forecasts = results.forecast(horizon=1, start = length-lag, method='simulation', simulations=1000)
    sims = forecasts.variance
    std = np.sqrt(np.sum(sims)*12/lag)
    return std

In [ ]:
def post_allocation(dfMVOweight):

    dfMVOweight['equal']=1/len(dfMVOweight)
    numofstocks=len(dfMVOweight)
    allocationweight=allocation.loc[numofstocks,:]
    dfMVOweight['proportion']=''
    dfMVOweight['adjusted proportion']=''
    dfMVOweight['direct']=''
    residual=0
    for i in range(numofstocks):
        dfMVOweight.iloc[i,4]=dfMVOweight.iloc[i,1]/np.sum(dfMVOweight['PR_SR'])
        if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
            dfMVOweight.iloc[i,5]=allocationweight[0]
        else:
            dfMVOweight.iloc[i,5]=(dfMVOweight.iloc[i,4]+residual/(numofstocks-i))
        residual+=dfMVOweight.iloc[i,4]-dfMVOweight.iloc[i,5]
    for i in range(int(allocationweight[1])):
        dfMVOweight.iloc[i,6]=allocationweight[0]
    for i in range(int(allocationweight[1]),int(allocationweight[1]+allocationweight[3])):
        dfMVOweight.iloc[i,6]=allocationweight[2]
    return dfMVOweight

In [ ]:
def output(weight):
    if stock_type == "S&P100":
      OEX=pdr.get_data_yahoo("^OEX", start=start_swan_date, end=end_swan_date)["Adj Close"]

    elif stock_type == "S&P500":
      OEX=pdr.get_data_yahoo("SPY", start=start_swan_date, end=end_swan_date)["Adj Close"]

    stock_price=pdr.get_data_yahoo(list(weight.index), start=start_swan_date, end=end_swan_date)["Adj Close"]
    OEX.to_csv("oex.csv")
    injected_result = pd.DataFrame(index = stock_price.index )
    S1=np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
    portfolio_value = stock_price@S1.T
    injected_result["portfolio_value"]=portfolio_value
    S2 = 10000/float(OEX[0])
    injected_result["oex"]=S2*OEX
    return injected_result

# Step 1 - Preselection

In [ ]:
# Preselection parameters set up
method = "K_means_multiple"          # or avg or anyother method
stock_type = "S&P100"
end_swan = "Swan8"         #equals the last swan before the swan we are going to predict
start_swan="Swan4"
start = dt(2019,1,1)
end = dt(2020,2,15)
selected_stock_number = 22     #number of stock preselected
start_swan_date = dt(2020,2,15)
end_swan_date = dt(2021,3,23)
w1 = 0.17
w2 = 0.17
MA_days = 30
N = 7 # This parameter is set to help classification method like K-means

In [ ]:
#read all stocks and their swan beta
if stock_type == "S&P100":
    portfolio_beta = pd.read_excel("All_SwanBeta100_MVO2.xlsx",index_col=0)
else:
    portfolio_beta = pd.read_excel("All_SwanBeta500.xlsx",index_col=0)

In [ ]:
selected_swans_var=portfolio_beta.loc[:,start_swan:end_swan]
portfolio_beta =portfolio_beta[ selected_swans_var.T.isnull().sum()<selected_swans_var.shape[1]-1]

In [ ]:
portfolio_beta

,Swan-3,Swan-2,Swan-1,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10a,Swan10b
Ticker,,,,,,,,,,,,,,
CHTR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.583170,0.649030,0.797374,0.939407,1.060138,1.278190
GM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.026711,0.792127,0.190040,1.553517,2.447747,1.934870
KMI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.540529,1.230239,0.919927,1.592887,-1.141071,-0.541752
GOOG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.803565,1.167641,0.916183,0.944312,0.976498,0.875866,1.104899
GOOGL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.803565,1.167641,0.870361,0.938163,0.979057,0.880849,1.117074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
UNP,0.850695,2.127809,1.993802,1.088334,0.427848,0.418409,0.283910,1.275539,1.334344,1.354688,1.049957,1.188887,-0.200038,0.632258
USB,0.407208,0.375769,0.483245,0.176170,1.187362,1.307068,0.811824,1.672467,0.905422,0.995170,0.906766,1.495656,0.501194,0.825853
WFC,1.388900,1.261192,1.686449,0.558622,1.587780,1.233240,0.300985,1.698834,1.182443,0.995686,0.827634,1.470901,0.571633,1.078228


In [ ]:
corr=portfolio_beta.corr()
selected_corr=corr.loc["Swan1":end_swan,"Swan1":end_swan]
corr1={}
for i in range(selected_corr.shape[1]):
    for j in range(i+1,selected_corr.shape[0]):
        selected_corr.iloc[j,i]=np.nan

In [ ]:
selected_corr

,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8
Swan1,1.0,-0.004268,0.111982,0.050073,0.001444,0.101963,0.079685,0.076826
Swan2,NaN,1.000000,0.549511,0.335485,0.410861,0.392490,0.033937,0.307858
Swan3,NaN,NaN,1.000000,0.151778,0.258652,0.435457,0.243705,0.224528
Swan4,NaN,NaN,NaN,1.000000,-0.032138,0.143125,0.147027,0.225411
Swan5,NaN,NaN,NaN,NaN,1.000000,0.444474,0.221512,0.208068
Swan6,NaN,NaN,NaN,NaN,NaN,1.000000,0.301317,0.482711
Swan7,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.185835
Swan8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000


In [ ]:
avg=selected_corr.mean(axis=1)
weight_swan = avg/sum(avg)
selected_beata=portfolio_beta.loc[:,start_swan:end_swan].fillna(0)
#portfolio_beta["corr_beta"]=selected_beata@weight_swan
#portfolio_beta=portfolio_beta.sort_values(by=["corr_beta"])

In [ ]:
portfolio_beta

,Swan-3,Swan-2,Swan-1,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10a,Swan10b
Ticker,,,,,,,,,,,,,,
CHTR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.583170,0.649030,0.797374,0.939407,1.060138,1.278190
GM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.026711,0.792127,0.190040,1.553517,2.447747,1.934870
KMI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.540529,1.230239,0.919927,1.592887,-1.141071,-0.541752
GOOG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.803565,1.167641,0.916183,0.944312,0.976498,0.875866,1.104899
GOOGL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.803565,1.167641,0.870361,0.938163,0.979057,0.880849,1.117074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
UNP,0.850695,2.127809,1.993802,1.088334,0.427848,0.418409,0.283910,1.275539,1.334344,1.354688,1.049957,1.188887,-0.200038,0.632258
USB,0.407208,0.375769,0.483245,0.176170,1.187362,1.307068,0.811824,1.672467,0.905422,0.995170,0.906766,1.495656,0.501194,0.825853
WFC,1.388900,1.261192,1.686449,0.558622,1.587780,1.233240,0.300985,1.698834,1.182443,0.995686,0.827634,1.470901,0.571633,1.078228


In [ ]:
if method == "gamma":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    portfolio_beta["avg_sd"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)+portfolio_beta.loc[:,start_swan:end_swan].std(axis=1,ddof=0)
    portfolio_beta=portfolio_beta.sort_values(by=["avg_sd"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]
elif method =="average":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    #Select 18 stocks
    portfolio_beta=portfolio_beta.sort_values(by=["avg"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]

elif method == "K_means":
    selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
    Y_predic = selected_beta[end_swan].copy()
    Kmean = KMeans(n_clusters=N,random_state=1)#set the random_state to keep modeling the same every run
    Kmean.fit(selected_beta)
    ct = Kmean.cluster_centers_
    labels = Kmean.labels_
    current_dbscore=metrics.davies_bouldin_score(selected_beta, labels)
    print(current_dbscore)
    selected_beta["Group"]=Kmean.labels_
    print(selected_beta['Group'])
    res = ct.mean(axis=1)
    res [res<0]=0

    min_group = np.where(res==min(res))
    for i in range(N):
        Y_predic[selected_beta["Group"] == i] = res [i]
    selected_beta ["avg"]=Y_predic.values
    selected_beta=selected_beta.sort_values(by=["avg"])
    first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
    if first_group.shape[0] <10:
        second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
        preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
    else:
        preselected_stock = first_group

# Use results from multiple clusters
elif method=="K_means_multiple":
    selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
    Y_predic = selected_beta[end_swan].copy()
    seed_dict={}

    # Run 1000 trials with davies bouldin index
    for r in range(1,1000):
      Kmean = KMeans(n_clusters=N,random_state=r,n_init=10)
      Kmean.fit(selected_beta)
      labels = Kmean.labels_
      # Evaluate the davies-bouldin score
      dbscore=metrics.davies_bouldin_score(selected_beta, labels)
      seed_dict[r]=dbscore
      sorted_dict=sorted(seed_dict.items(), key=lambda x:x[1])

    # Put all stocks into a stock list
    stock_list=[]
    max_length=0
    for index in range(0,5):
      selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
      Kmean = KMeans(n_clusters=N,random_state=sorted_dict[index][0],n_init=10)
      Kmean.fit(selected_beta)
      ct = Kmean.cluster_centers_
      Kmean.labels_
      selected_beta["Group"]=Kmean.labels_
      res = ct.mean(axis=1)
      res [res<0]=0
      min_group = np.where(res==min(res))
      for i in range(N):
          Y_predic[selected_beta["Group"] == i] = res [i]
      selected_beta ["avg"]=Y_predic.values
      selected_beta=selected_beta.sort_values(by=["avg"])
      first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
      if first_group.shape[0] <10:
          second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
          preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
      else:
          preselected_stock = first_group
      if preselected_stock.shape[0]>max_length:
        max_length=preselected_stock.shape[0]
      stock_list.extend(preselected_stock.index.values)
    print(stock_list)

    # Count the times each stock appear
    values,counts=np.unique(stock_list,return_counts=True)
    count_dict={}
    preselected_stock=[]
    for t in range(0,len(values)):
      count_dict[values[t]]=counts[t]
    sorted_count=sorted(count_dict.items(), key=lambda x:x[1],reverse=True)
    print(sorted_count)

    d = {'stock':[], 'frequency':[]}
    df = pd.DataFrame(data=d)
    for item in sorted_count:
      df.loc[len(df.index)] = [item[0],item[1]]
    df.to_csv('stock_frequency.csv')

    for p in range(0,min(22,len(sorted_count))):
      if sorted_count[p][1]>=0:
        preselected_stock.append(sorted_count[p][0])

    preselected_stock = sorted(preselected_stock)
    selected_beta=selected_beta.loc[preselected_stock]
    selected_beta["avg"]=selected_beta.loc[:,start_swan:end_swan].mean(axis=1)
    #selected_beta["avg"]=selected_beta["avg"].mean()
    preselected_stock = selected_beta
    print(preselected_stock)

['LLY', 'SBUX', 'NEE', 'PEP', 'VZ', 'PG', 'ABT', 'AMZN', 'SO', 'MRK', 'JNJ', 'MDLZ', 'KO', 'WMT', 'CL', 'EXC', 'CL', 'LLY', 'VZ', 'AMGN', 'KO', 'MO', 'COST', 'MRK', 'NEE', 'ABT', 'SBUX', 'PEP', 'PG', 'PFE', 'SO', 'WMT', 'MDLZ', 'JNJ', 'EXC', 'MO', 'EXC', 'VZ', 'AMGN', 'JNJ', 'KO', 'LLY', 'COST', 'WMT', 'SPG', 'SBUX', 'ABT', 'MRK', 'SO', 'NEE', 'AMZN', 'PEP', 'PFE', 'PG', 'CL', 'MDLZ', 'BRK-B', 'CL', 'VZ', 'AMGN', 'KO', 'MO', 'COST', 'MRK', 'NEE', 'ABT', 'SBUX', 'PEP', 'SO', 'PG', 'PFE', 'MDLZ', 'WMT', 'JNJ', 'EXC', 'LLY', 'CL', 'SO', 'AMGN', 'LLY', 'VZ', 'MO', 'COST', 'MRK', 'NEE', 'KO', 'SBUX', 'PEP', 'PFE', 'PG', 'JNJ', 'AMZN', 'EXC', 'MDLZ', 'WMT', 'ABT']
[('ABT', 5), ('CL', 5), ('EXC', 5), ('JNJ', 5), ('KO', 5), ('LLY', 5), ('MDLZ', 5), ('MRK', 5), ('NEE', 5), ('PEP', 5), ('PG', 5), ('SBUX', 5), ('SO', 5), ('VZ', 5), ('WMT', 5), ('AMGN', 4), ('COST', 4), ('MO', 4), ('PFE', 4), ('AMZN', 3), ('BRK-B', 1), ('SPG', 1)]
           Swan4     Swan5     Swan6     Swan7     Swan8  Group    

<ipython-input-57-212a9c64d9f6>:71: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
<ipython-input-57-212a9c64d9f6>:71: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
<ipython-input-57-212a9c64d9f6>:71: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_b

In [ ]:
ct

array([[ 1.25271924,  0.76441264,  0.85093361,  0.89894274,  0.99578754],
       [ 1.15801823,  1.53335123,  1.91648284,  1.20046569,  1.31333892],
       [ 0.41888824,  0.66417188,  0.24686177,  0.64299047,  0.3006769 ],
       [ 1.81120562,  0.55906473, -0.04732315,  0.43355465, -0.29862099],
       [ 0.3268658 ,  1.19327404,  1.11764369,  0.71405383,  0.92461757],
       [ 1.45708836,  0.44989021,  0.28989564,  1.63984707,  1.35436378],
       [ 2.54565626,  0.63571696,  1.25106235, -0.25340007,  2.78743002]])

In [ ]:
res

array([0.95255915, 1.42433138, 0.45471785, 0.49157617, 0.85529099,
       1.03821701, 1.39329311])

In [ ]:
preselected_stock.info()

<class 'pandas.core.frame.DataFrame'>
Index: 22 entries, ABT to WMT
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Swan4   22 non-null     float64
 1   Swan5   22 non-null     float64
 2   Swan6   22 non-null     float64
 3   Swan7   22 non-null     float64
 4   Swan8   22 non-null     float64
 5   Group   22 non-null     int32  
 6   avg     22 non-null     float64
dtypes: float64(6), int32(1)
memory usage: 1.3+ KB


In [ ]:
#portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan]*weight

In [ ]:
#order = list(preselected_stock.index)
#random.shuffle(order)
#preselected_stock = preselected_stock.reindex(index = order)
preselected_stock

,Swan4,Swan5,Swan6,Swan7,Swan8,Group,avg
Ticker,,,,,,,
ABT,0.702188,0.474696,0.311079,1.073150,0.431921,2,0.598607
AMGN,0.795355,0.502892,0.109292,0.843475,0.659977,2,0.582198
AMZN,-0.445606,0.328167,0.123350,0.334033,1.561997,2,0.380388
BRK-B,0.220942,1.172896,0.533684,0.870710,0.663086,4,0.692263
CL,0.099788,0.617542,-0.045657,0.784583,0.569564,2,0.405164
COST,0.829818,0.976294,-0.015790,0.618966,0.918643,2,0.665586
EXC,0.519667,0.794675,0.263389,0.605246,-0.123438,2,0.411908
JNJ,0.403161,0.739963,0.360346,0.668742,0.555245,2,0.545491
KO,-0.170562,0.536675,0.325968,0.614677,-0.049526,2,0.251446


In [ ]:
preselected_stock["PR"]=1-preselected_stock["avg"]
preselected_stock.to_csv('preselected_stocks.csv')
preselected_stock

,Swan4,Swan5,Swan6,Swan7,Swan8,Group,avg,PR
Ticker,,,,,,,,
ABT,0.702188,0.474696,0.311079,1.073150,0.431921,2,0.598607,0.401393
AMGN,0.795355,0.502892,0.109292,0.843475,0.659977,2,0.582198,0.417802
AMZN,-0.445606,0.328167,0.123350,0.334033,1.561997,2,0.380388,0.619612
BRK-B,0.220942,1.172896,0.533684,0.870710,0.663086,4,0.692263,0.307737
CL,0.099788,0.617542,-0.045657,0.784583,0.569564,2,0.405164,0.594836
COST,0.829818,0.976294,-0.015790,0.618966,0.918643,2,0.665586,0.334414
EXC,0.519667,0.794675,0.263389,0.605246,-0.123438,2,0.411908,0.588092
JNJ,0.403161,0.739963,0.360346,0.668742,0.555245,2,0.545491,0.454509
KO,-0.170562,0.536675,0.325968,0.614677,-0.049526,2,0.251446,0.748554


# Step 2 - Download data and adjuste price based on Swan Beta

In [ ]:
#next line is used to input preselected stock from excel, if you are going to preselect by your self, use this blocks and note next one.
#selected_stock_list =  pd.read_excel("19_stickers.xlsx",header = None).values[:,0]
#swan_beta=pd.read_excel("19_stickers.xlsx",header = None,index_col=0).iloc[:,0]

In [ ]:
#Set the paramter to download tradeinfo from Yahoo Finance
selected_stock=preselected_stock
selected_stock_list=selected_stock.index.values
swan_beta=preselected_stock.loc[selected_stock_list,"avg"]
print(selected_stock_list)

['ABT' 'AMGN' 'AMZN' 'BRK-B' 'CL' 'COST' 'EXC' 'JNJ' 'KO' 'LLY' 'MDLZ'
 'MO' 'MRK' 'NEE' 'PEP' 'PFE' 'PG' 'SBUX' 'SO' 'SPG' 'VZ' 'WMT']


In [ ]:
#download data from yahoo finance
yahooDataFrame= pdr.get_data_yahoo(list(selected_stock_list), start=start, end=end)["Adj Close"]


[*********************100%%**********************]  22 of 22 completed


In [ ]:
yahooDataFrame.to_csv('yahooDataFrame.csv')
yahooDataFrame=yahooDataFrame[selected_stock_list]
yahooDataFrame

Ticker,ABT,AMGN,AMZN,BRK-B,CL,COST,EXC,JNJ,KO,LLY,...,MRK,NEE,PEP,PFE,PG,SBUX,SO,SPG,VZ,WMT
Date,,,,,,,,,,,,,,,,,,,,,
2019-01-02,62.657768,159.867599,76.956497,202.800003,51.385071,186.350311,25.484903,109.023033,39.253029,105.295570,...,60.730312,36.857189,92.703285,31.940178,78.474579,56.738705,34.275482,119.271179,40.790031,28.355381
2019-01-03,59.700672,157.435013,75.014000,191.660004,51.142235,182.400513,25.513821,107.290596,39.010468,102.023415,...,59.485023,36.766041,91.838005,31.046589,77.924370,54.277565,34.761551,119.169998,40.935654,28.209560
2019-01-04,61.404613,162.816711,78.769501,195.199997,51.619221,187.697235,25.802963,109.091293,39.788330,105.093941,...,61.276646,37.443169,93.721252,31.755554,79.514832,56.077110,35.051620,120.984268,41.037586,28.385759
2019-01-07,62.324184,165.007706,81.475502,196.910004,51.931438,188.388885,25.849220,108.391510,39.269756,105.662201,...,60.601772,37.388905,92.915352,31.925413,79.196739,56.077110,34.973225,122.256416,41.299725,28.719923
2019-01-08,61.575916,167.123734,82.829002,196.309998,52.512505,189.799561,26.248243,110.909073,39.713058,106.633759,...,61.051678,37.662350,93.806091,32.073120,79.489029,56.121208,35.968880,125.176559,42.508427,28.920424
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-02-10,81.270958,195.220047,106.695503,227.110001,69.203598,288.990509,28.818163,133.251968,51.559425,136.381073,...,70.695503,60.400383,127.230881,29.255289,112.183105,79.029427,56.103107,108.752586,46.195110,35.720406
2020-02-11,82.071686,196.542023,107.540001,227.330002,68.094193,286.126343,29.432589,133.366074,51.421364,135.200150,...,70.357124,60.491516,127.597763,29.471937,110.395912,79.011391,56.226063,108.007607,45.006489,35.766891
2020-02-12,81.721947,194.859528,108.000000,227.440002,67.951042,290.215454,29.384855,132.576370,51.266037,134.506577,...,68.706528,60.071438,127.597763,29.201134,109.764633,79.968445,56.250652,105.795448,44.653736,35.906364


In [ ]:
#set parameters to calculate predict market drop
market_drop_info = pd.read_excel("MarketDD.xlsx",index_col=0)
#swan_to_predict=100
total_day=yahooDataFrame.shape[0]
t1=int(round(market_drop_info.loc[:end_swan,"T1"].mean(),0))
t1=int((np.array(market_drop_info.loc[:end_swan,"T1"])*np.array(weight_swan)).sum())
#t2=total_day-t1
t2=int(round(market_drop_info.loc[:end_swan,"T2"].mean(),0))
dd=market_drop_info.loc[:end_swan,"Actual DD"].mean()
t0=int(total_day-t1-t2)

In [ ]:
#calculate predict market drop
md = [] #Make market drop data list named md


j = 0 #Start from 0
inc = dd/(t1-1) #Calculate unit increasing market drop
for i in range(t1): #Append market drop data for t1
    md.append(round(j,4))
    j = j + inc

base = (t2+1)**(1/(dd*100)) #Calculate log base
l1 = [] #Make decreasing market drop list
for i in range(t2):
    l1.append(math.log((i+2),base)/100)

for i in range(t2): #Append market drop data for t2
    md.append(round((dd-l1[i]),4))

for i in range(t0):
    md.append(0)

market_drop = pd.DataFrame(md) #Change list into dataframe
#market_drop = market_drop.rename(columns={0: 'Swan Drop spy'}) #Change column name
market_drop1 = pd.read_csv("trainingMarketdrop.csv")

In [ ]:
market_drop.to_csv("market_drop.csv")

In [ ]:
market_drop

,0
0,0.0000
1,0.0037
2,0.0074
3,0.0111
4,0.0148
...,...
278,0.0000
279,0.0000
280,0.0000
281,0.0000


In [ ]:
swan_drop_pct=np.array(swan_beta)*np.array(market_drop)
yahooDataFrame=pd.DataFrame((1-swan_drop_pct)*np.array(yahooDataFrame),index=yahooDataFrame.index,columns=yahooDataFrame.columns)

In [ ]:
yahooDataFrame=yahooDataFrame.dropna(how="all",axis = 1)
selected_stock_list=yahooDataFrame.columns.values
selected_stock_number = yahooDataFrame.shape[1]
selected_stock_list

array(['ABT', 'AMGN', 'AMZN', 'BRK-B', 'CL', 'COST', 'EXC', 'JNJ', 'KO',
       'LLY', 'MDLZ', 'MO', 'MRK', 'NEE', 'PEP', 'PFE', 'PG', 'SBUX',
       'SO', 'SPG', 'VZ', 'WMT'], dtype=object)

In [ ]:
yahooDataFrame.to_csv('yahooDataFrame.csv')
yahooDataFrame

Ticker,ABT,AMGN,AMZN,BRK-B,CL,COST,EXC,JNJ,KO,LLY,...,MRK,NEE,PEP,PFE,PG,SBUX,SO,SPG,VZ,WMT
Date,,,,,,,,,,,,,,,,,,,,,
2019-01-02,62.657768,159.867599,76.956497,202.800003,51.385071,186.350311,25.484903,109.023033,39.253029,105.295570,...,60.730312,36.857189,92.703285,31.940178,78.474579,56.738705,34.275482,119.271179,40.790031,28.355381
2019-01-03,59.568444,157.095877,74.908423,191.169091,51.065567,181.951321,25.474936,107.074049,38.974174,101.812828,...,59.351865,36.725252,91.673367,30.967518,77.815943,54.192028,34.756037,118.919555,40.872652,28.153629
2019-01-04,61.132610,162.115254,78.547775,194.200036,51.464455,186.772763,25.724313,108.650932,39.714296,104.660092,...,61.002307,37.360090,93.385224,31.593801,79.293551,55.900364,35.040500,120.475757,40.911269,28.273197
2019-01-07,61.910069,163.941360,81.131488,195.396923,51.697886,186.997067,25.731033,107.735204,39.160152,105.007909,...,60.194796,37.264466,92.415644,31.681485,78.866147,55.811991,34.956581,121.485629,41.109038,28.549092
2019-01-08,61.030392,165.683707,82.362697,194.298704,52.197617,187.929907,26.088228,110.013674,39.565270,105.753348,...,60.505014,37.495219,93.133426,31.746378,79.046612,55.767438,35.946056,124.124296,42.246738,28.691059
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-02-10,81.270958,195.220047,106.695503,227.110001,69.203598,288.990509,28.818163,133.251968,51.559425,136.381073,...,70.695503,60.400383,127.230881,29.255289,112.183105,79.029427,56.103107,108.752586,46.195110,35.720406
2020-02-11,82.071686,196.542023,107.540001,227.330002,68.094193,286.126343,29.432589,133.366074,51.421364,135.200150,...,70.357124,60.491516,127.597763,29.471937,110.395912,79.011391,56.226063,108.007607,45.006489,35.766891
2020-02-12,81.721947,194.859528,108.000000,227.440002,67.951042,290.215454,29.384855,132.576370,51.266037,134.506577,...,68.706528,60.071438,127.597763,29.201134,109.764633,79.968445,56.250652,105.795448,44.653736,35.906364


# Step3 - MVO

In [ ]:
#Set the paramter for MVO process
Portfolio_Date=end

In [ ]:
#data cleaning - filling missing data in yahooDataFrame
data_cleaning(yahooDataFrame)

In [ ]:
#calculated rf
#bond_yield = quandl.get('FRED/DGS3MO',api_key='X4EQzCpMfeaTCTDthRH_')
#bond_yield = quandl.get('FRED/DGS3MO')
#bond_yield=np.log(bond_yield/100+1)
#date=list(map(str,bond_yield.index.values))
#date=list(map(modify_date,date))
#week=pd.to_datetime(date).strftime("%W").values
#week_1=np.roll(week,1)
#week_difference=np.where(week!=week_1)
#week_difference=np.delete(week_difference,0,None)
#rf_week=bond_yield.iloc[week_difference-1]

In [ ]:
# Calculated R for 1 year and 0.5 year
today_price = yahooDataFrame.iloc[-MA_days:,:]
one_year_ago_data = get_price_years_ago(yahooDataFrame,end,1)
one_year_ago_date = one_year_ago_data.index.values[-1]
half_year_ago_data = get_price_years_ago(yahooDataFrame,end,0.5)
half_year_ago_date = half_year_ago_data.index.values[-1]
one_year_ago_price=yahooDataFrame.loc[one_year_ago_date:,:]
one_year_ago_price=yahooDataFrame.iloc[:MA_days,:]
half_year_ago_price=yahooDataFrame.loc[half_year_ago_date:,:]
half_year_ago_price=yahooDataFrame.iloc[:MA_days,:]

R_mv_one_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(one_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)
R_mv_half_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(half_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)



In [ ]:
# Calculate MA
MA_Data=yahooDataFrame.rolling(MA_days).mean().dropna(axis=0)
MA_Data=MA_Data.loc[one_year_ago_date:,:]
MA_now = MA_Data.iloc[-1,:]
MA_one_year_ago = MA_Data.iloc[0,:]
MA_half_year_ago = MA_Data.loc[half_year_ago_date,:]

In [ ]:
# Calculate daily return
daily_return = yahooDataFrame.pct_change().dropna()
log_daily_return = np.log(1+daily_return)

# Get week end stock price data
stock_date = yahooDataFrame.index.values
week=pd.to_datetime(stock_date).strftime("%W").values
week_1=np.roll(week,1)
week_difference=np.where(week!=week_1)
week_difference=np.delete(week_difference,0,None)
# Calculate weekly return

weekly_return=yahooDataFrame.iloc[week_difference-1].pct_change().dropna()
log_weekly_return = np.log(1+weekly_return)

# Get month end stock price data
stock_date = yahooDataFrame.index.values
month=pd.to_datetime(stock_date).strftime("%m").values
month_1=np.roll(month,1)
month_difference=np.where(month!=month_1)
month_difference=np.delete(month_difference,0,None)
# Calculate monthly return
monthly_return=yahooDataFrame.iloc[month_difference-1].pct_change().dropna()
log_monthly_return = np.log(1+monthly_return)

# Get semi-month end stock price data
semi_month = []
for i in range(1,len(month_difference)):
    semi_month.append((month_difference[i]-month_difference[i-1])//2+month_difference[i-1])
    semi_month.append(month_difference[i]-1)
# Calculate semi-monthly return
semi_monthly_return=yahooDataFrame.iloc[semi_month].pct_change().dropna()
log_semi_monthly_return = np.log(1+semi_monthly_return)

In [ ]:
# Calculated Max Dropdown for each stock
stock_return_one_year=daily_return.loc[one_year_ago_date:,:]
mdd=[]
for column in stock_return_one_year.columns:
    mdd .append(max_drawdown_absolute(stock_return_one_year.loc[:,column]))
mdd=pd.DataFrame(mdd,index = stock_return_one_year.columns,columns=["MAXDD","start_index","end_index"])
mar=R_mv_one_year/mdd.iloc[:,0]

In [ ]:
#calculate weekly sd for each stock
one_year_weekly_return=log_weekly_return.iloc[log_weekly_return.index >= one_year_ago_date]
weekly_std= np.std(one_year_weekly_return)*np.sqrt(one_year_weekly_return.shape[0])

#Calculate garch_std sd for each stock
one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
res_all = one_year_monthly_return - np.mean(one_year_monthly_return)
judge = one_year_monthly_return.shape[0]
for stock in one_year_monthly_return.columns:
    res=res_all[stock]
    h= het_arch(res)
    if judge < 100 or h ==0:
        garch_std = weekly_std

    else:#build Garch Model
        std = []
        if judge <=200:
            p = 0
            q = 6
        else:
            p = 1
            q = 1

        std.append(fit_garch(one_year_monthly_return[stock],p,q))
        garch_std = pd.DataFrame(std,index = one_year_monthly_return.columns)

#Calculate monthly sd for each stock
one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
monthly_std = np.std(one_year_monthly_return)*np.sqrt(12);

#Calculate semi_monthly sd for each stock
one_year_semi_monthly_return=log_semi_monthly_return.iloc[log_semi_monthly_return.index >= one_year_ago_date]
semi_monthly_std = np.std(one_year_semi_monthly_return)*np.sqrt(24);


/usr/local/lib/python3.10/dist-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


In [ ]:
# Calculated MaxDrop
DD_range = 90
total_range = 90
DD_end = yahooDataFrame.iloc[yahooDataFrame.index <= one_year_ago_date]
DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
max_DD=[-1]*selected_stock_number


if (DD_start.shape[0] != 1 ) and (DD_end.shape[0] != 1):
    DD_start = DD_start.index[-1]
    DD_end = DD_end.index[-1]
    while DD_start > DD_end :
        DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
        DD_mid = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(DD_range+total_range)]
        if (DD_start.shape[0] != 1 ) and (DD_mid.shape[0] != 1):
            DD_start = DD_start.index[-1]
            DD_mid = DD_mid.index[-1]
            price = yahooDataFrame.loc[DD_start:DD_mid,:]
            for i in range(len(yahooDataFrame.columns)):
                max_DD[i]=max(max_DD[i],max_drawdown_absolute(stock_return_one_year.iloc[:,i])[0])
            total_range = total_range + 1
        else:
            break

In [ ]:
# Read OPT-data
optdata = pd.read_excel("optdata_MVO2.xlsx",index_col=0,sheet_name = 1)
P_current = MA_now
P_fwd = optdata.P_fwd.loc[P_current.index]
R_fwd=P_fwd/P_current-1
R_60 = P_current/MA_one_year_ago - 1
R_60_half = P_current/MA_half_year_ago - 1
R_MA_1ago = R_mv_one_year
R_MA_halfago = R_mv_half_year
Rfwd_adjusted = pd.Series(index=R_MA_1ago.index)
for i in R_fwd.index:
    if R_fwd[i] < 0.5*R_60[i]:
        Rfwd_adjusted[i] = R_60_half[i]
    elif R_fwd[i] > 1.3*R_60[i]:
        if 1.3*R_60[i] > 0.7*R_fwd[i]:
            Rfwd_adjusted[i] = 1.3*R_60[i]
        else:
            Rfwd_adjusted[i] = 0.7*R_fwd[i]
    else:
        Rfwd_adjusted[i] = R_fwd[i]
Rfwd_adjusted[Rfwd_adjusted<0]=0

In [ ]:
price_now = today_price.iloc[-1,:]
optimizer = optdata.loc[P_current.index,:]
FFC = pd.DataFrame([R_MA_1ago.values,R_MA_halfago.values,MA_now.values,price_now.values,semi_monthly_std.values,weekly_std.values,MA_one_year_ago.values,MA_half_year_ago.values,max_DD,R_60.values,R_60_half.values,P_fwd.values,Rfwd_adjusted.values],columns = R_MA_1ago.index,index = ["R_MA_1ago","R_MA_halfago","MA_now","price",'std','std_f','MA_1ago','MA_halfago','Fall_MDD','R_60','R_60_half','R_fwd',"Rfwd_adjusted"]).T
preselected_stock = preselected_stock.loc[selected_stock_list,:]
preselected_stock["Rfwd_adjusted"]=Rfwd_adjusted.values
preselected_stock["std_f"]=weekly_std.values
preselected_stock["SR"]=preselected_stock["Rfwd_adjusted"]/preselected_stock["std_f"]
preselected_stock["PR_SR"]=preselected_stock["PR"]*preselected_stock["SR"]**0.5
preselected_stock.to_csv("preselected_Stocks_info.csv")
p=FFC.MA_now
EPS_fwd=optimizer.EPS_fwd
EPS_ttm=optimizer.EPS_ttm
fall=FFC.Fall_MDD
FFC.to_csv("WEDataBank.csv")

PE=p/EPS_fwd
PEG= p/(EPS_fwd*(EPS_fwd/EPS_ttm-1))
PEG[PEG<0]=2000


In [ ]:
cov = pd.DataFrame(np.cov(np.exp(one_year_weekly_return)-1,rowvar=False,bias = False)*51,index= one_year_weekly_return.columns, columns= one_year_weekly_return.columns)
E_2 = Rfwd_adjusted/fall
E_3 = Rfwd_adjusted/PEG
E_4 = Rfwd_adjusted
std =  np.std(np.exp(one_year_weekly_return)-1)*np.sqrt(52)


/usr/local/lib/python3.10/dist-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


# Optimization

In [ ]:
w0 = 1/selected_stock_number*np.ones(selected_stock_number)
uplim_ini = np.zeros(selected_stock_number)
uplim3 =pd.DataFrame( w1*(1-uplim_ini) + w2 * uplim_ini,index = std.index,columns=["weight"])
for i in uplim3.index:
    if i in ["SPY","IWD"]:
        uplim3.loc[i,:]=0
low_lim =  np.zeros(selected_stock_number)
multi=lambda x: x.T @cov@x

In [ ]:
ef = EfficientFrontier(E_2/100, cov,weight_bounds=(0, w1))
#ef.add_constraint(lambda w: w@E_2== 12)
raw_weights = ef.max_sharpe(0.01)
cleaned_weights = ef.clean_weights()
print(cleaned_weights)
ef.portfolio_performance(verbose=True)


OrderedDict([('ABT', 0.0), ('AMGN', 0.01896), ('AMZN', 0.17), ('BRK-B', 0.0255), ('CL', 0.0), ('COST', 0.05576), ('EXC', 0.0), ('JNJ', 0.0), ('KO', 0.03231), ('LLY', 0.10798), ('MDLZ', 0.0), ('MO', 0.06829), ('MRK', 0.00514), ('NEE', 0.17), ('PEP', 0.0), ('PFE', 0.0), ('PG', 0.0), ('SBUX', 0.0), ('SO', 0.17), ('SPG', 0.00606), ('VZ', 0.0), ('WMT', 0.17)])
Expected annual return: 29.8%
Annual volatility: 7.6%
Sharpe Ratio: 3.79


/usr/local/lib/python3.10/dist-packages/pypfopt/efficient_frontier/efficient_frontier.py:441: UserWarning: The risk_free_rate provided to portfolio_performance is different to the one used by max_sharpe. Using the previous value.
  warnings.warn(


(0.2980729793499889, 0.07593213735895625, 3.793821553951167)

In [ ]:
weight1=np.array(pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"]))[:,0]

In [ ]:
sd1 = np.sqrt(weight1@cov@weight1.T)
R1 = weight1.T@E_2/100
sharpe_ratio1 = (R1-0.01)/sd1
print("new_sharpe:",sharpe_ratio1)

new_sharpe: 3.7938215532247224


weight2 =   pd.read_excel("old_weight.xlsx",index_col=0)
weight2=np.array(weight2.loc[list(cleaned_weights),:])[:,0]

sd2 = np.sqrt(weight2@cov@weight2.T)
R2 = weight2.T@E_2/100
sharpe_ratio2 = (R2-0.01)/sd2
print("old_sharpe:",sharpe_ratio2)

In [ ]:
weight=pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"])
weight = weight[weight.weight >=0.030]
weight.weight = weight.weight/sum(weight.weight)
weight.to_csv("weights.csv")

See pyportfolioopt.readthedocs.io/en/latest/MeanVariance.html for more reference


In [ ]:
if stock_type == "S&P100":
    OEX=pdr.get_data_yahoo("^OEX", start=start_swan_date, end=end_swan_date)["Adj Close"]

elif stock_type == "S&P500":
    OEX=pdr.get_data_yahoo("SPY", start=start_swan_date, end=end_swan_date)["Adj Close"]

stock_price=pdr.get_data_yahoo(list(weight.index), start=start_swan_date, end=end_swan_date)["Adj Close"]

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  8 of 8 completed


In [ ]:
stock_price.to_csv("tradeinfo.csv")
stock_price=stock_price[weight.index]
stock_price

Ticker,AMZN,COST,KO,LLY,MO,NEE,SO,WMT
Date,,,,,,,,
2020-02-18,107.783501,296.597656,51.369583,132.848450,30.615488,62.689804,57.401833,37.077938
2020-02-19,108.511002,297.490997,51.576683,132.678864,30.561323,61.051651,56.748417,36.473549
2020-02-20,107.654999,298.467346,51.533539,133.781128,30.845678,61.576210,57.740940,36.476646
2020-02-21,104.798500,296.505646,51.887341,133.818787,31.069101,61.700687,57.062714,36.752491
2020-02-24,100.464500,288.833984,50.610218,130.153961,29.654114,60.889385,56.078449,36.052032
...,...,...,...,...,...,...,...,...
2021-03-16,154.593002,311.866150,46.091713,183.273483,36.585869,68.760590,52.944149,42.032837
2021-03-17,156.786499,313.714935,46.109718,179.270096,36.837479,66.553116,52.468739,41.683052
2021-03-18,151.399506,307.796906,45.506786,176.581894,37.422085,64.883827,52.209423,41.138798


In [ ]:
weight

,weight
AMZN,0.180020
COST,0.059047
KO,0.034214
LLY,0.114344
MO,0.072315
NEE,0.180020
SO,0.180020
WMT,0.180020


In [ ]:
start=stock_price.loc["2020-02-19"]
end = stock_price.loc["2020-03-23"]
stock_drop=abs(end/start-1)
market_drop = abs(OEX.loc["2020-03-23"]/OEX.loc["2020-02-19"]-1)

In [ ]:
beta = stock_drop/market_drop
1-beta.T@weight

,0
weight,0.32561


In [ ]:
beta

,0
Ticker,
AMZN,0.390752
COST,0.368082
KO,1.161009
LLY,0.490481
MO,0.966755
NEE,1.062980
SO,1.173178
WMT,0.078002


In [ ]:
stock_drop

,0
Ticker,
AMZN,0.123209
COST,0.116061
KO,0.366080
LLY,0.154655
MO,0.304829
NEE,0.335170
SO,0.369917
WMT,0.024595


In [ ]:
rest=1
extra=0
for i, row in weight.iterrows():
  if row['weight']>0.17:
    rest=rest-row['weight']
    extra=extra+(row['weight']-0.17)
    # row['weight']=0.17

for i,row in weight.iterrows():
  if row['weight']<0.165:
    # row['weight']=row['weight']+row['weight']/rest*extra
    a=1

weight.to_csv("weights.csv")
weight

,weight
AMZN,0.180020
COST,0.059047
KO,0.034214
LLY,0.114344
MO,0.072315
NEE,0.180020
SO,0.180020
WMT,0.180020


In [ ]:
portfolio_beta = portfolio_beta.loc[weight.index]
portfolio_beta

,Swan-3,Swan-2,Swan-1,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10a,Swan10b
AMZN,NaN,NaN,NaN,NaN,NaN,1.584517,-0.445606,0.328167,0.123350,0.334033,1.561997,0.390752,1.440979,1.690943
COST,NaN,NaN,NaN,1.052794,1.853865,1.404504,0.829818,0.976294,-0.015790,0.618966,0.918643,0.368083,0.536185,0.824934
KO,0.722917,-0.425109,-0.480807,1.142373,0.525786,1.283611,-0.170562,0.536675,0.325968,0.614677,-0.049526,1.161009,0.077061,-0.058129
LLY,0.963250,0.201971,-0.343427,1.143233,0.898003,0.125934,0.746268,0.920577,0.280671,0.780342,0.061466,0.490481,0.281552,-0.331040
MO,0.308320,0.177528,0.542427,0.449382,0.538171,-0.274878,0.818853,0.448863,-0.137637,0.264024,1.030036,0.966755,-0.434630,-0.002853
NEE,0.253300,-0.134129,0.533954,0.579706,0.166846,-0.420653,0.414534,0.503894,0.415367,0.192033,-0.026635,1.062981,0.721034,0.894444
SO,NaN,NaN,0.692450,0.735964,-0.130113,-0.265152,-0.138738,0.669547,-0.279810,0.055182,-0.091814,1.173178,-0.036447,0.004823
WMT,0.897873,-1.424131,0.462878,1.059818,1.484265,0.762944,0.586029,0.461023,0.235666,0.982186,0.414455,0.078002,0.291725,0.710631


In [ ]:
a=stock_price@weight
b=1-a.loc["2020-03-23"]/a.loc["2020-02-19"]
1-b/market_drop

,0
weight,0.40602


In [ ]:
injected_result = pd.DataFrame(index = stock_price.index)
S1 = np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T

In [ ]:
portfolio_value = np.sum(np.array(weight.T) * stock_price,axis=1)
S1 = np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
injected_result["portfolio_value"]=stock_price@S1.T
S2 = 10000/float(OEX[0])
injected_result["oex"]=S2*OEX
injected_result.to_csv("Injected_result.csv")


<ipython-input-103-fb66288e77f1>:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 10000/float(OEX[0])


In [ ]:
temp=injected_result.loc[["2020-02-19","2020-03-23"]]
Portfolio_DD=abs(temp['portfolio_value'][1]/temp['portfolio_value'][0]-1)
DD=abs(temp['oex'][1]/temp['oex'][0]-1)
PR=1-(Portfolio_DD/DD)
print(PR)
temp

0.3270649354074948


<ipython-input-104-0cde1dbf7bf2>:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Portfolio_DD=abs(temp['portfolio_value'][1]/temp['portfolio_value'][0]-1)
<ipython-input-104-0cde1dbf7bf2>:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  DD=abs(temp['oex'][1]/temp['oex'][0]-1)


,portfolio_value,oex
Date,,
2020-02-19,9915.692175,10051.749340
2020-03-23,7811.736195,6882.312479


In [ ]:
allocation=pd.read_excel('allocation weight.xlsx',index_col='stock selected')
dfMVOstock=deepcopy(weight)
dfMVOstock['PR_SR']=''
dfMVOstock['SR']=''
for i in range(len(weight)):
    dfMVOstock.iloc[i,1]=preselected_stock.loc[weight.index[i],'PR_SR']
    dfMVOstock.iloc[i,2]=preselected_stock.loc[weight.index[i],'SR']
dfMVOstock_sort=dfMVOstock.sort_values(by='PR_SR',ascending=False)
postallocationweight=post_allocation(dfMVOstock_sort)
postallocationweight


<ipython-input-11-2b9c13413cb0>:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
<ipython-input-11-2b9c13413cb0>:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dfMVOweight.iloc[i,5]=allocationweight[0]
<ipython-input-11-2b9c13413cb0>:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
<i

,weight,PR_SR,SR,equal,proportion,adjusted proportion,direct
AMZN,0.180020,5.79676,87.524744,0.125,0.415921,0.17,0.17
NEE,0.180020,3.028258,18.706363,0.125,0.217279,0.17,0.17
WMT,0.180020,1.666579,12.893634,0.125,0.119578,0.168445,0.17
SO,0.180020,1.276483,1.778652,0.125,0.091588,0.140455,0.17
MO,0.072315,0.643466,1.560081,0.125,0.046169,0.095036,0.17
KO,0.034214,0.641488,0.734397,0.125,0.046027,0.094894,0.05
LLY,0.114344,0.557813,1.59172,0.125,0.040023,0.08889,0.05
COST,0.059047,0.326308,0.95211,0.125,0.023413,0.07228,0.05


In [ ]:
#draw type E graph for postallocation
MVOweight=postallocationweight['weight']
equalweight=postallocationweight['equal']
proportionweight=postallocationweight['adjusted proportion']
directweight=postallocationweight['direct']
output(MVOweight).to_csv('no_post_allocation_result.csv')
output(equalweight).to_csv('equal_weight_result.csv')
output(proportionweight).to_csv('proportional_weight_result.csv')
output(directweight).to_csv('direct_weight_result.csv')

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  8 of 8 completed
<ipython-input-12-01d80541063f>:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 10000/float(OEX[0])
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  8 of 8 completed
<ipython-input-12-01d80541063f>:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 10000/float(OEX[0])
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  8 of 8 comple

#